# Lab 3: Generador de Imágenes con CrewAI

Sistema multi-agente usando **CrewAI** — una tripulación (crew) de agentes con roles, tareas, y herramientas.

**Arquitectura (con archivos de configuración YAML):**
- `config/agents.yaml` — define los agentes: `writer`, `reviewer`, `image_creator`
- `config/tasks.yaml` — define las tareas: `write_task`, `review_task`, `image_task`
- `crew.py` — clase `@CrewBase` que conecta agentes, tareas y herramientas

**Diferencias con Lab 2 (OpenAI Agents SDK):**
- CrewAI usa Crews (tripulaciones) con Tasks secuenciales o jerárquicas
- Los agentes se configuran en YAML con roles, backstory, y goals explícitos
- Las herramientas se definen con `@tool` y se asignan en la clase Crew
- El flujo se define como `Process.sequential` o `Process.hierarchical`

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

for key in ("OPENAI_API_KEY", "GOOGLE_API_KEY"):
    val = os.getenv(key)
    if val:
        os.environ[key] = val
        print(f"{key}: {val[:8]}...{val[-4:]}")
    else:
        print(f"{key} not found.")

In [ ]:
# Importar la Crew desde el módulo con archivos de configuración
import sys
sys.path.insert(0, "src")

from lab3_crewai.crew import ImageGeneratorCrew, last_image_path, gemini_client, GEMINI_IMAGE_MODEL, IMAGE_GEN_CONFIG
import lab3_crewai.crew as crew_module

print("Estructura del proyecto CrewAI:")
print("  src/lab3_crewai/")
print("  ├── config/")
print("  │   ├── agents.yaml    ← define writer, reviewer, image_creator")
print("  │   └── tasks.yaml     ← define write_task, review_task, image_task")
print("  └── crew.py            ← @CrewBase clase + @tool generate_image")

In [ ]:
from crewai import Agent, Task, Crew, Process

# --- Definición de Agentes ---

writer = Agent(
    role="Escritor Técnico",
    goal="Crear descripciones técnicas detalladas y visuales de conceptos complejos",
    backstory=(
        "Eres un comunicador técnico experto con 10 años de experiencia "
        "creando documentación visual. Tu especialidad es convertir conceptos "
        "abstractos en descripciones que cualquiera pueda visualizar como un diagrama."
    ),
    verbose=True,
    llm="gpt-4.1-nano",
)

reviewer = Agent(
    role="Revisor Senior",
    goal="Asegurar que las descripciones son completas, claras y tienen suficiente detalle visual",
    backstory=(
        "Eres un líder técnico senior con ojo para el detalle. Tu trabajo es "
        "revisar descripciones y enriquecerlas — nunca minimizar. Si algo falta, "
        "lo añades. Si algo es vago, lo haces concreto."
    ),
    verbose=True,
    llm="gpt-4.1-mini",
)

image_creator = Agent(
    role="Creador de Imágenes",
    goal="Llamar a la herramienta generate_image con el texto aprobado para generar la imagen",
    backstory=(
        "Eres un agente que SIEMPRE usa la herramienta generate_image. "
        "Recibes texto aprobado y tu ÚNICA acción es llamar a generate_image "
        "pasando ese texto como argumento approved_text. "
        "NUNCA respondas con texto — siempre llama a la herramienta."
    ),
    verbose=True,
    llm="gpt-4.1-mini",
    tools=[generate_image],
)

print("Agentes CrewAI definidos:")
print(f"  writer: {writer.role}")
print(f"  reviewer: {reviewer.role}")
print(f"  image_creator: {image_creator.role} (con herramienta generate_image)")

In [ ]:
# --- Definición de Tareas con contexto explícito ---

def create_crew(topic: str) -> Crew:
    """Crea una Crew con tareas encadenadas para un tema dado."""

    write_task = Task(
        description=(
            f"Escribe una descripción técnica detallada sobre: {topic}\n\n"
            "Incluye:\n"
            "- Los componentes o actores principales involucrados\n"
            "- Cómo interactúan o se conectan\n"
            "- La secuencia o flujo de operaciones\n"
            "- Detalles visuales que ayuden a dibujar un diagrama\n\n"
            "Escribe 3-5 frases detalladas. Sé específico y descriptivo."
        ),
        expected_output="Una descripción técnica detallada de 3-5 frases con componentes, flujos y detalles visuales.",
        agent=writer,
    )

    review_task = Task(
        description=(
            "Revisa y enriquece la descripción técnica que recibiste del escritor.\n\n"
            "Verifica que incluye:\n"
            "1. Todos los componentes clave están nombrados y descritos\n"
            "2. Las relaciones y flujos de datos son explícitos\n"
            "3. La secuencia de operaciones es clara\n"
            "4. Hay suficiente detalle visual para dibujar un diagrama\n\n"
            "Si falta algo, añádelo. Devuelve la versión mejorada completa (al menos 4-6 frases)."
        ),
        expected_output="El texto mejorado y enriquecido con todos los detalles necesarios para visualización.",
        agent=reviewer,
        context=[write_task],
    )

    image_task = Task(
        description=(
            "Usa la herramienta generate_image para crear una imagen estilo whiteboard.\n"
            "Pasa como argumento 'approved_text' el texto completo que recibiste del revisor.\n"
            "DEBES llamar a la herramienta generate_image. No respondas con texto descriptivo."
        ),
        expected_output="La ruta al archivo de imagen generado (empieza con /tmp/ o /var/).",
        agent=image_creator,
        context=[review_task],
    )

    return Crew(
        agents=[writer, reviewer, image_creator],
        tasks=[write_task, review_task, image_task],
        process=Process.sequential,
        verbose=True,
    )


print("Función create_crew definida con context entre tareas:")
print("  write_task → review_task (context) → image_task (context + tool)")